In [1]:
# !wget https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

In [3]:
# # Download GloVe (one time, ~1 min)
# ! wget http://nlp.stanford.edu/data/glove.6B.zip
# ! unzip glove.6B.zip


In [4]:
"""
SQuAD Answer Generation with GloVe Embeddings + Q/K Hypothesis Testing

EXPECTED PERFORMANCE:
- With GloVe embeddings: 40-55% F1 ✓
- Training time: ~40-50 minutes
- Can reach 50%+ with Q/K hypothesis
"""

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from transformers import GPT2Tokenizer
import json
from collections import Counter
import string
import re
from tqdm import tqdm
import numpy as np
import os
import urllib.request
import zipfile

# Configuration
TEST_QK_HYPOTHESIS = False  # Set True after baseline completes
QK_LR_MULTIPLIER = 2.5  # Q/K learn 2.5x faster

# Optimized for GloVe embeddings
D_MODEL = 300  # Match GloVe dimension exactly
N_HEADS = 6
N_LAYERS = 6
D_FF = 1200
MAX_SEQ_LEN = 256
MAX_ANSWER_LEN = 50
DROPOUT = 0.2
BATCH_SIZE = 32
ACCUMULATION_STEPS = 2  # Effective batch: 48
BASE_LR = 5e-4
WARMUP_STEPS = 1000
NUM_EPOCHS = 70
GRAD_CLIP = 0.5
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.1
TRAIN_SUBSET_SIZE = 60000  # More data with GloVe
VAL_SUBSET_SIZE = 10000


def download_and_extract_glove():
    """Download and extract GloVe embeddings"""
    glove_file = 'glove.6B.300d.txt'
    
    if os.path.exists(glove_file):
        print(f"✓ GloVe embeddings found: {glove_file}")
        return glove_file
    
    print("\n" + "="*70)
    print("DOWNLOADING GLOVE EMBEDDINGS")
    print("="*70)
    
    zip_file = 'glove.6B.zip'
    
    if not os.path.exists(zip_file):
        print("Downloading GloVe 6B (822MB)... This may take a few minutes")
        url = 'https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip'
        
        try:
            # Download with progress bar
            response = urllib.request.urlopen(url)
            total_size = int(response.headers.get('content-length', 0))
            
            with open(zip_file, 'wb') as f, tqdm(
                total=total_size, unit='B', unit_scale=True, desc='Downloading'
            ) as pbar:
                while True:
                    chunk = response.read(8192)
                    if not chunk:
                        break
                    f.write(chunk)
                    pbar.update(len(chunk))
            
            print("✓ Download complete!")
        except Exception as e:
            print(f"Download failed: {e}")
            print("\nAlternative: Download manually from:")
            print("  https://nlp.stanford.edu/projects/glove/")
            print("  or https://huggingface.co/stanfordnlp/glove")
            return None
    
    # Extract
    if os.path.exists(zip_file):
        print("Extracting GloVe embeddings...")
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            # Only extract the 300d file we need
            zip_ref.extract('glove.6B.300d.txt')
        print("✓ Extraction complete!")
        
        # Optionally remove zip to save space
        # os.remove(zip_file)
    
    if os.path.exists(glove_file):
        return glove_file
    else:
        print("⚠ GloVe file not found after extraction")
        return None


def load_glove_embeddings(glove_file, tokenizer, embedding_dim=300):
    """Load GloVe and create embedding matrix for GPT-2 tokenizer"""
    print("\n" + "="*70)
    print("LOADING GLOVE EMBEDDINGS")
    print("="*70)
    
    # Load GloVe vectors
    print("Reading GloVe file (this takes ~1 minute)...")
    glove_vectors = {}
    
    with open(glove_file, 'r', encoding='utf-8') as f:
        for line in tqdm(f, total=400000, desc="Loading GloVe"):
            values = line.rstrip().split(' ')
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            glove_vectors[word] = vector
    
    print(f"✓ Loaded {len(glove_vectors):,} GloVe vectors")
    
    # Create embedding matrix for tokenizer vocabulary
    vocab_size = tokenizer.vocab_size
    embedding_matrix = np.random.normal(0, 0.1, (vocab_size, embedding_dim)).astype('float32')
    
    # Match tokenizer vocab with GloVe
    print("Matching tokenizer vocabulary with GloVee alpha finding is really significant - it suggests transformers might naturally want to do error correction but standard architectures prevent it!...")
    matched = 0
    
    for token, idx in tqdm(tokenizer.get_vocab().items(), desc="Matching"):
        # Try different matching strategies
        token_clean = token.replace('Ġ', '').replace('Ċ', '').lower().strip()
        
        if token in glove_vectors:
            embedding_matrix[idx] = glove_vectors[token]
            matched += 1
        elif token.lower() in glove_vectors:
            embedding_matrix[idx] = glove_vectors[token.lower()]
            matched += 1
        elif token_clean in glove_vectors:
            embedding_matrix[idx] = glove_vectors[token_clean]
            matched += 1
        # For subword tokens, try averaging character embeddings
        elif len(token_clean) > 0 and all(c.isalpha() for c in token_clean):
            # Use random but consistent embedding for unknown tokens
            pass
    
    match_rate = 100 * matched / vocab_size
    print(f"✓ Matched {matched:,}/{vocab_size:,} tokens ({match_rate:.1f}%)")
    print("="*70 + "\n")
    
    return torch.FloatTensor(embedding_matrix)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.last_attention_weights = None
        
    def forward(self, q, k, v, mask=None, save_attention=False):
        bs = q.size(0)
        
        q = self.q_linear(q).view(bs, -1, self.n_heads, self.d_k).transpose(1, 2)
        k = self.k_linear(k).view(bs, -1, self.n_heads, self.d_k).transpose(1, 2)
        v = self.v_linear(v).view(bs, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.d_k ** 0.5)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn = torch.softmax(scores, dim=-1)
        if save_attention:
            self.last_attention_weights = attn.detach()
        
        attn = self.dropout(attn)
        context = torch.matmul(attn, v)
        context = context.transpose(1, 2).contiguous().view(bs, -1, self.n_heads * self.d_k)
        
        return self.out(context)


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
    def forward(self, x, mask=None, save_attention=False):
        # Pre-norm
        attn_out = self.self_attn(self.norm1(x), self.norm1(x), self.norm1(x), mask, save_attention)
        x = x + attn_out
        x = x + self.ff(self.norm2(x))
        return x


class GPTAnswerGenerator(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, max_seq_len, dropout=0.1, pretrained_embeddings=None):
        super().__init__()
        
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        
        # Initialize with pretrained embeddings if provided
        if pretrained_embeddings is not None:
            print("Initializing token embeddings with GloVe...")
            self.token_embedding.weight.data.copy_(pretrained_embeddings)
            print("✓ Token embeddings initialized with GloVe")
        
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        
        self.norm = nn.LayerNorm(d_model)
        self.output = nn.Linear(d_model, vocab_size)
        
        # Weight tying
        self.output.weight = self.token_embedding.weight
        
        # Initialize non-embedding weights
        self._init_weights()
        
    def _init_weights(self):
        # Don't reinitialize token_embedding if using GloVe
        for name, p in self.named_parameters():
            if 'token_embedding' not in name and p.dim() > 1:
                nn.init.xavier_uniform_(p, gain=1/np.sqrt(2))
        
    def forward(self, x, mask=None, save_attention=False):
        pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        x = self.token_embedding(x) + self.position_embedding(pos)
        x = self.emb_dropout(x)
        
        for layer in self.layers:
            x = layer(x, mask, save_attention)
        
        return self.output(self.norm(x))
    
    def get_attention_weights(self):
        return [layer.self_attn.last_attention_weights for layer in self.layers]


class SQuADDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_len, max_ans_len):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.max_ans_len = max_ans_len
        self.data = []
        
        with open(data_path, 'r') as f:
            squad = json.load(f)
        
        for article in squad['data']:
            for para in article['paragraphs']:
                ctx = para['context']
                for qa in para['qas']:
                    if not qa['is_impossible'] and qa['answers']:
                        ans = qa['answers'][0]['text']
                        ans_start = qa['answers'][0]['answer_start']
                        
                        # Extract relevant context window
                        start = max(0, ans_start - 200)
                        end = min(len(ctx), ans_start + len(ans) + 200)
                        focused_ctx = ctx[start:end]
                        
                        self.data.append({
                            'context': focused_ctx,
                            'question': qa['question'],
                            'answer': ans
                        })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Format: Q: question C: context A: answer
        prefix = f"Q: {item['question']} C: {item['context']} A:"
        answer = f" {item['answer']}"
        
        prefix_ids = self.tokenizer.encode(prefix, max_length=self.max_len-self.max_ans_len-2, 
                                          truncation=True, add_special_tokens=False)
        answer_ids = self.tokenizer.encode(answer, max_length=self.max_ans_len, 
                                          truncation=True, add_special_tokens=False)
        answer_ids.append(self.tokenizer.eos_token_id)
        
        input_ids = prefix_ids + answer_ids
        labels = [-100] * len(prefix_ids) + answer_ids
        
        # Pad
        while len(input_ids) < self.max_len:
            input_ids.append(self.tokenizer.pad_token_id)
            labels.append(-100)
        
        return {
            'input_ids': torch.tensor(input_ids[:self.max_len]),
            'labels': torch.tensor(labels[:self.max_len])
        }


def create_mask(seq_len, device):
    return (torch.triu(torch.ones(seq_len, seq_len, device=device), 1) == 0).unsqueeze(0).unsqueeze(0)


def normalize_answer(s):
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(c for c in s if c not in string.punctuation)
    return ' '.join(s.split())


def f1_score(pred, truth):
    pred_tok = normalize_answer(pred).split()
    truth_tok = normalize_answer(truth).split()
    
    if not pred_tok or not truth_tok:
        return int(pred_tok == truth_tok)
    
    common = Counter(pred_tok) & Counter(truth_tok)
    if not common:
        return 0
    
    prec = sum(common.values()) / len(pred_tok)
    rec = sum(common.values()) / len(truth_tok)
    return 2 * prec * rec / (prec + rec)


def exact_match(pred, truth):
    return int(normalize_answer(pred) == normalize_answer(truth))


def train_epoch(model, loader, opt, sched, device, epoch):
    model.train()
    total_loss = 0
    opt.zero_grad()
    
    pbar = tqdm(loader, desc=f"Epoch {epoch}")
    for i, batch in enumerate(pbar):
        inp = batch['input_ids'].to(device)
        lbl = batch['labels'].to(device)
        
        mask = create_mask(inp.size(1), device)
        logits = model(inp, mask)
        
        # Shift for next-token prediction
        loss = nn.functional.cross_entropy(
            logits[:, :-1].reshape(-1, logits.size(-1)),
            lbl[:, 1:].reshape(-1),
            ignore_index=-100,
            label_smoothing=LABEL_SMOOTHING
        )
        
        loss = loss / ACCUMULATION_STEPS
        loss.backward()
        
        if (i + 1) % ACCUMULATION_STEPS == 0:
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            sched.step()
            opt.zero_grad()
        
        total_loss += loss.item() * ACCUMULATION_STEPS
        pbar.set_postfix({'loss': f'{loss.item() * ACCUMULATION_STEPS:.3f}'})
    
    return total_loss / len(loader)


def generate(model, tokenizer, context, question, device, max_len=50):
    model.eval()
    
    prompt = f"Q: {question} C: {context} A:"
    ids = tokenizer.encode(prompt, max_length=MAX_SEQ_LEN-max_len-5, 
                          truncation=True, add_special_tokens=False, return_tensors='pt').to(device)
    
    start_len = ids.size(1)
    
    with torch.no_grad():
        for _ in range(max_len):
            if ids.size(1) >= MAX_SEQ_LEN:
                break
            
            mask = create_mask(ids.size(1), device)
            logits = model(ids, mask)
            next_tok = logits[:, -1].argmax(-1, keepdim=True)
            ids = torch.cat([ids, next_tok], 1)
            
            if next_tok.item() == tokenizer.eos_token_id:
                break
    
    return tokenizer.decode(ids[0, start_len:], skip_special_tokens=True).strip()


def evaluate(model, dataset, tokenizer, device, n_samples=300):
    model.eval()
    f1_sum = em_sum = 0
    
    if isinstance(dataset, Subset):
        items = [dataset.dataset.data[dataset.indices[i]] for i in range(min(n_samples, len(dataset)))]
    else:
        items = dataset.data[:n_samples]
    
    for item in tqdm(items, desc="Eval"):
        pred = generate(model, tokenizer, item['context'], item['question'], device)
        f1_sum += f1_score(pred, item['answer'])
        em_sum += exact_match(pred, item['answer'])
    
    return {'f1': f1_sum / len(items), 'em': em_sum / len(items)}


def analyze_attention(model, dataset, tokenizer, device, n=30):
    model.eval()
    scores = []
    
    if isinstance(dataset, Subset):
        items = [dataset.dataset.data[dataset.indices[i]] for i in range(min(n, len(dataset)))]
    else:
        items = dataset.data[:n]
    
    for item in items:
        prompt = f"Q: {item['question']} C: {item['context']} A:"
        ids = tokenizer.encode(prompt, max_length=MAX_SEQ_LEN-MAX_ANSWER_LEN, 
                              truncation=True, add_special_tokens=False, return_tensors='pt').to(device)
        
        with torch.no_grad():
            mask = create_mask(ids.size(1), device)
            model(ids, mask, save_attention=True)
            
            weights = model.get_attention_weights()
            if weights[0] is not None:
                avg = torch.stack([w[0] for w in weights if w is not None]).mean(0)
                scores.append(avg[0].mean().item())
    
    return np.mean(scores) if scores else 0


if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print("="*70)
    print("SQUAD ANSWER GENERATION WITH GLOVE EMBEDDINGS")
    print("="*70)
    #print(f"Expected F1: 40-55% (with GloVe)")
    print(f"Model: {N_LAYERS}L, {D_MODEL}d, {N_HEADS}h")
    print(f"Device: {device}")
    print("="*70 + "\n")
    
    # Download and load GloVe
    glove_file = download_and_extract_glove()
    
    if glove_file is None:
        print("\n WARNING: Could not load GloVe embeddings")
        print("Proceeding without pretrained embeddings (expect 15-25% F1)")
        pretrained_embeddings = None
    
    # Download SQuAD datasets
    for name in ['train-v2.0.json', 'dev-v2.0.json']:
        if not os.path.exists(name):
            print(f"Downloading {name}...")
            urllib.request.urlretrieve(
                f'https://rajpurkar.github.io/SQuAD-explorer/dataset/{name}', name)
    
    # Setup tokenizer
    print("Loading tokenizer...")
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token
    
    # Load GloVe embeddings for tokenizer
    if glove_file:
        pretrained_embeddings = load_glove_embeddings(glove_file, tokenizer, D_MODEL)
    else:
        pretrained_embeddings = None
    
    # Load datasets
    print("Loading datasets...")
    full_train = SQuADDataset('train-v2.0.json', tokenizer, MAX_SEQ_LEN, MAX_ANSWER_LEN)
    full_val = SQuADDataset('dev-v2.0.json', tokenizer, MAX_SEQ_LEN, MAX_ANSWER_LEN)
    
    train_ds = Subset(full_train, torch.randperm(len(full_train))[:TRAIN_SUBSET_SIZE])
    val_ds = Subset(full_val, torch.randperm(len(full_val))[:VAL_SUBSET_SIZE])
    
    print(f"Train: {len(train_ds)}, Val: {len(val_ds)}\n")
    
    loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    
    # Model
    print("Initializing model...")
    n_seed = [1234,1235,1236,1237,1238]
    for seed_ in n_seed:
        print("Training for seed", seed_)
        torch.manual_seed(seed_)
        model = GPTAnswerGenerator(
            vocab_size=tokenizer.vocab_size,
            d_model=D_MODEL,
            n_heads=N_HEADS,
            n_layers=N_LAYERS,
            d_ff=D_FF,
            max_seq_len=MAX_SEQ_LEN,
            dropout=DROPOUT,
            pretrained_embeddings=pretrained_embeddings
        ).to(device)

        total_params = sum(p.numel() for p in model.parameters()) / 1e6
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
        print(f"Total parameters: {total_params:.1f}M")
        print(f"Trainable parameters: {trainable_params:.1f}M\n")

        # Optimizer with differential learning rates for embeddings
        if TEST_QK_HYPOTHESIS:
            print("="*70)
            print(f"TESTING Q/K HYPOTHESIS - Q/K LR = {QK_LR_MULTIPLIER}x")
            print("="*70 + "\n")

            qk = [p for n, p in model.named_parameters() if 'q_linear' in n or 'k_linear' in n]
            other = [p for n, p in model.named_parameters() if 'q_linear' not in n and 'k_linear' not in n]

            print(f"Q/K params: {sum(p.numel() for p in qk)/1e6:.1f}M")
            print(f"Other params: {sum(p.numel() for p in other)/1e6:.1f}M\n")

            opt = torch.optim.AdamW([
                {'params': qk, 'lr': BASE_LR * QK_LR_MULTIPLIER, 'weight_decay': WEIGHT_DECAY},
                {'params': other, 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY}
            ])

            sched = torch.optim.lr_scheduler.OneCycleLR(
                opt, [BASE_LR * QK_LR_MULTIPLIER, BASE_LR],
                total_steps=len(loader) * NUM_EPOCHS,
                pct_start=WARMUP_STEPS / (len(loader) * NUM_EPOCHS)
            )
        else:
            print("="*70)
            print("BASELINE (Standard LR)")
            print("="*70 + "\n")

            # Use lower LR for pretrained embeddings if they exist
            if pretrained_embeddings is not None:
                embedding_params = [model.token_embedding.weight]
                other_params = [p for n, p in model.named_parameters() if 'token_embedding' not in n]

                opt = torch.optim.AdamW([
                    {'params': embedding_params, 'lr': BASE_LR * 0.1, 'weight_decay': 0},  # Fine-tune slowly
                    {'params': other_params, 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY}
                ])

                print("Using differential LR: embeddings=0.1x, other=1.0x\n")
            else:
                opt = torch.optim.AdamW(model.parameters(), BASE_LR, weight_decay=WEIGHT_DECAY)

            sched = torch.optim.lr_scheduler.OneCycleLR(
                opt,
                max_lr=BASE_LR if pretrained_embeddings is None else [BASE_LR * 0.1, BASE_LR],
                total_steps=len(loader) * NUM_EPOCHS,
                pct_start=WARMUP_STEPS / (len(loader) * NUM_EPOCHS)
            )

        # Train
        best_f1 = 0
        results = {'loss': [], 'train_f1': [], 'val_f1': [], 'val_em': [], 'attn': []}

        for e in range(NUM_EPOCHS):
            print(f"\n{'='*70}")
            print(f"EPOCH {e+1}/{NUM_EPOCHS}")
            print('='*70)

            loss = train_epoch(model, loader, opt, sched, device, e+1)
            results['loss'].append(loss)
            print(f"\nLoss: {loss:.4f}")

            # Eval
            train_m = evaluate(model, train_ds, tokenizer, device, 200)
            val_m = evaluate(model, val_ds, tokenizer, device, 300)

            results['train_f1'].append(train_m['f1'])
            results['val_f1'].append(val_m['f1'])
            results['val_em'].append(val_m['em'])

            gap = train_m['f1'] - val_m['f1']
            print(f"Train F1: {train_m['f1']:.4f} | Val F1: {val_m['f1']:.4f} | Gap: {gap:.4f} | EM: {val_m['em']:.4f}")

            # Sample
            if e % 20 == 0:
                item = val_ds.dataset.data[val_ds.indices[0]]
                pred = generate(model, tokenizer, item['context'], item['question'], device)
                print(f"\nSample:")
                print(f"  Q: {item['question'][:60]}...")
                print(f"  True: {item['answer']}")
                print(f"  Pred: {pred}")
                print(f"  F1: {f1_score(pred, item['answer']):.3f}")

            # Attention
            if e % 20 == 0 and TEST_QK_HYPOTHESIS:
                attn = analyze_attention(model, val_ds, tokenizer, device)
                results['attn'].append(attn)
                print(f"Attention: {attn:.4f}")

            # Save best
            if val_m['f1'] > best_f1:
                best_f1 = val_m['f1']
                name = 'best_qk_'+str(seed_)+'.pt' if TEST_QK_HYPOTHESIS else 'best_baseline_'+str(seed_)+'.pt'
                torch.save({'model': model.state_dict(), 'f1': best_f1, 'epoch': e+1}, name)
                print(f"✓ SAVED! Best F1: {best_f1:.4f}")

        # Final
        print(f"\n{'='*70}")
        print("FINAL RESULTS")
        print('='*70)
        print(f"Best Val F1: {best_f1*100:.1f}%")
        print(f"Final Val F1: {results['val_f1'][-1]*100:.1f}%")
        print(f"Final EM: {results['val_em'][-1]*100:.1f}%")
        print(f"Train-Val Gap: {results['train_f1'][-1] - results['val_f1'][-1]:.4f}")


SQUAD ANSWER GENERATION WITH GLOVE EMBEDDINGS
Model: 6L, 300d, 6h
Device: cuda

✓ GloVe embeddings found: glove.6B.300d.txt
Loading tokenizer...

LOADING GLOVE EMBEDDINGS
Reading GloVe file (this takes ~1 minute)...


Loading GloVe: 100%|█████████████████| 400000/400000 [00:21<00:00, 18609.31it/s]


✓ Loaded 400,000 GloVe vectors
Matching tokenizer vocabulary with GloVee alpha finding is really significant - it suggests transformers might naturally want to do error correction but standard architectures prevent it!...


Matching: 100%|███████████████████████| 50257/50257 [00:00<00:00, 332747.38it/s]

✓ Matched 43,058/50,257 tokens (85.7%)

Loading datasets...


...

Train F1: 0.9417 | Val F1: 0.5767 | Gap: 0.3650 | EM: 0.3733

EPOCH 70/70


Epoch 70: 100%|█████████████████| 1875/1875 [05:03<00:00,  6.17it/s, loss=1.844]



Loss: 1.8731


Eval: 100%|███████████████████████████████████| 300/300 [00:08<00:00, 33.63it/s]

Train F1: 0.9415 | Val F1: 0.5912 | Gap: 0.3502 | EM: 0.3600

FINAL RESULTS
Best Val F1: 59.2%
Final Val F1: 59.1%
Final EM: 36.0%
Train-Val Gap: 0.3502


In [5]:
# Eval
train_m = evaluate(model, train_ds, tokenizer, device, 40000)

Eval: 100%|███████████████████████████████| 40000/40000 [22:35<00:00, 29.52it/s]


In [ ]:
train_m["f1"],train_m["em"]

In [ ]:
val_m = evaluate(model, val_ds, tokenizer, device, 20000)

In [ ]:
val_m["f1"],val_m["em"]

In [ ]:
"""
Simple Analysis Functions for Notebook

IMPORTANT: Your dataset needs answer span information!

First, reload your validation dataset with this code:
    val_dataset_full = load_squad_with_spans('dev-v2.0.json', tokenizer)
    
Then run analysis:
    results = analyze_model(model, val_dataset_full, tokenizer, device, n_samples=100)
"""

import torch
import numpy as np
from tqdm import tqdm
from collections import Counter
import string
import re
import json


def load_squad_with_spans(data_path, tokenizer):
    """
    Load SQuAD dataset with answer span information
    This is needed for sufficiency, faithfulness, and attention analysis
    """
    class SQuADWithSpans:
        def __init__(self, data_path):
            self.data = []
            
            with open(data_path, 'r', encoding='utf-8') as f:
                squad = json.load(f)
            
            for article in squad['data']:
                for para in article['paragraphs']:
                    ctx = para['context']
                    for qa in para['qas']:
                        if not qa['is_impossible'] and qa['answers']:
                            ans = qa['answers'][0]['text']
                            ans_start = qa['answers'][0]['answer_start']
                            ans_end = ans_start + len(ans)
                            
                            # Extract focused context
                            start = max(0, ans_start - 200)
                            end = min(len(ctx), ans_start + len(ans) + 200)
                            focused_ctx = ctx[start:end]
                            
                            # Adjust answer positions
                            adjusted_ans_start = ans_start - start
                            adjusted_ans_end = adjusted_ans_start + len(ans)
                            
                            self.data.append({
                                'context': focused_ctx,
                                'question': qa['question'],
                                'answer': ans,
                                'answer_start': adjusted_ans_start,
                                'answer_end': adjusted_ans_end
                            })
        
        def __len__(self):
            return len(self.data)
    
    return SQuADWithSpans(data_path)


def normalize_answer(s):
    """Normalize answer for comparison"""
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(c for c in s if c not in string.punctuation)
    return ' '.join(s.split())


def f1_score(pred, truth):
    """Compute F1 score"""
    pred_tok = normalize_answer(pred).split()
    truth_tok = normalize_answer(truth).split()
    
    if not pred_tok or not truth_tok:
        return int(pred_tok == truth_tok)
    
    common = Counter(pred_tok) & Counter(truth_tok)
    if not common:
        return 0
    
    prec = sum(common.values()) / len(pred_tok)
    rec = sum(common.values()) / len(truth_tok)
    return 2 * prec * rec / (prec + rec)


def create_mask(seq_len, device):
    """Create causal mask"""
    return (torch.triu(torch.ones(seq_len, seq_len, device=device), 1) == 0).unsqueeze(0).unsqueeze(0)


def generate(model, tokenizer, context, question, device, max_len=50):
    """Generate answer"""
    model.eval()
    
    prompt = f"Q: {question} C: {context} A:"
    ids = tokenizer.encode(prompt, max_length=320-max_len-5, 
                          truncation=True, add_special_tokens=False, return_tensors='pt').to(device)
    
    start_len = ids.size(1)
    
    with torch.no_grad():
        for _ in range(max_len):
            if ids.size(1) >= 320:
                break
            
            mask = create_mask(ids.size(1), device)
            logits = model(ids, mask)
            next_tok = logits[:, -1].argmax(-1, keepdim=True)
            ids = torch.cat([ids, next_tok], 1)
            
            if next_tok.item() == tokenizer.eos_token_id:
                break
    
    return tokenizer.decode(ids[0, start_len:], skip_special_tokens=True).strip()


def get_answer_token_positions(context, answer_start, answer_end, tokenizer):
    """
    Get token indices that correspond to answer span in context
    """
    tokens = tokenizer.tokenize(context)
    token_to_char = []
    
    current_pos = 0
    for token in tokens:
        # Handle GPT-2 special characters
        token_text = token.replace('Ġ', ' ').replace('Ċ', '\n')
        start_pos = context.find(token_text.strip(), current_pos)
        if start_pos == -1:
            start_pos = current_pos
        end_pos = start_pos + len(token_text.strip())
        token_to_char.append((start_pos, end_pos))
        current_pos = end_pos
    
    # Find tokens that overlap with answer span
    answer_token_indices = []
    for idx, (tok_start, tok_end) in enumerate(token_to_char):
        if not (tok_end <= answer_start or tok_start >= answer_end):
            answer_token_indices.append(idx)
    
    return answer_token_indices


def get_item_with_spans(dataset, idx):
    """
    Get item with answer span information from dataset
    Works with both Subset and regular Dataset
    """
    from torch.utils.data import Subset
    
    if isinstance(dataset, Subset):
        # Get original index from subset
        original_idx = dataset.indices[idx]
        item = dataset.dataset.data[original_idx]
    else:
        item = dataset.data[idx]
    
    # Check if item has span information
    if 'answer_start' not in item:
        # Need to reconstruct from original dataset
        # This happens if you're using the old dataset format
        return None
    
    return item


def compute_sufficiency(model, tokenizer, item, device):
    """
    Sufficiency: Can model answer correctly using ONLY the answer span?
    
    Returns: F1 score when using only answer span as context
    """
    # Check if we have span information
    if 'answer_start' not in item or 'answer_end' not in item:
        return None
    
    # Extract only the answer span
    answer_span = item['context'][item['answer_start']:item['answer_end']]
    
    # If span is empty, return None
    if not answer_span.strip():
        return None
    
    # Generate with only answer span as context
    pred = generate(model, tokenizer, answer_span, item['question'], device)
    
    # Compare to ground truth
    return f1_score(pred, item['answer'])


def compute_faithfulness(model, tokenizer, item, device):
    """
    Faithfulness: Does model's prediction change when answer span is removed?
    
    Returns: F1 drop when answer is removed (higher = more faithful)
    """
    # Check if we have span information
    if 'answer_start' not in item or 'answer_end' not in item:
        return None
    
    # Generate with full context
    pred_with = generate(model, tokenizer, item['context'], item['question'], device)
    f1_with = f1_score(pred_with, item['answer'])
    
    # Generate with answer span removed
    context_without = (item['context'][:item['answer_start']] + 
                      " [REMOVED] " + 
                      item['context'][item['answer_end']:])
    pred_without = generate(model, tokenizer, context_without, item['question'], device)
    f1_without = f1_score(pred_without, item['answer'])
    
    # Faithfulness = performance drop
    faithfulness = f1_with - f1_without
    
    return faithfulness


def compute_attention_on_answer_tokens(model, tokenizer, item, device):
    """
    Compute average attention score on answer token positions during generation
    
    Returns: Mean attention score on answer tokens
    """
    # Check if we have span information
    if 'answer_start' not in item or 'answer_end' not in item:
        return None
    
    model.eval()
    
    # Build prompt
    prompt = f"Q: {item['question']} C: {item['context']} A:"
    prompt_ids = tokenizer.encode(prompt, max_length=270, 
                                  truncation=True, add_special_tokens=False, 
                                  return_tensors='pt').to(device)
    
    # Get answer token positions in the context
    answer_tokens = get_answer_token_positions(
        item['context'], 
        item['answer_start'], 
        item['answer_end'], 
        tokenizer
    )
    
    if len(answer_tokens) == 0:
        return None  # No answer tokens found
    
    # Adjust positions for prompt structure "Q: ... C: ..."
    q_prefix = f"Q: {item['question']} C: "
    q_tokens_len = len(tokenizer.encode(q_prefix, add_special_tokens=False))
    answer_positions = [q_tokens_len + idx for idx in answer_tokens]
    
    # Generate and track attention
    generated = prompt_ids
    attention_scores = []
    
    with torch.no_grad():
        for _ in range(50):  # Max answer length
            if generated.size(1) >= 320:
                break
            
            mask = create_mask(generated.size(1), device)
            logits = model(generated, mask, save_attention=True)
            
            # Get attention weights from all layers
            attn_weights = model.get_attention_weights()
            
            if attn_weights[0] is not None:
                # Average across all layers and heads
                avg_attn = torch.stack([w[0] for w in attn_weights if w is not None]).mean(0)
                
                # Get attention from last generated token to all input tokens
                last_token_attn = avg_attn[0, -1, :]  # Shape: [seq_len]
                
                # Extract attention scores on answer token positions
                answer_attn = []
                for pos in answer_positions:
                    if pos < last_token_attn.size(0):
                        answer_attn.append(last_token_attn[pos].item())
                
                if answer_attn:
                    attention_scores.append(np.mean(answer_attn))
            
            # Generate next token
            next_tok = logits[:, -1].argmax(-1, keepdim=True)
            generated = torch.cat([generated, next_tok], 1)
            
            if next_tok.item() == tokenizer.eos_token_id:
                break
    
    # Return mean attention score across generation steps
    return np.mean(attention_scores) if attention_scores else None


def analyze_model(model, dataset, tokenizer, device, n_samples=100):
    """
    Run comprehensive analysis on model
    
    Args:
        model: Trained GPT model
        dataset: SQuAD dataset (or Subset) - must have answer_start/answer_end
        tokenizer: GPT2Tokenizer
        device: torch device
        n_samples: Number of samples to analyze
    
    Returns:
        dict with results and aggregate metrics
    """
    model.eval()
    
    # Handle Subset
    from torch.utils.data import Subset
    if isinstance(dataset, Subset):
        items = [dataset.dataset.data[dataset.indices[i]] 
                for i in range(min(n_samples, len(dataset)))]
    else:
        items = dataset.data[:n_samples]
    
    # Check if first item has span information
    if len(items) > 0 and 'answer_start' not in items[0]:
        print("\n" + "="*70)
        print("ERROR: Dataset does not have answer span information!")
        print("="*70)
        print("\nYour dataset items need 'answer_start' and 'answer_end' keys.")
        print("Please reload your dataset with the updated SQuADDataset class that")
        print("includes span information.")
        print("\nTo fix: Reload val_dataset using the code from the training script")
        print("that includes answer_start and answer_end in the data.")
        print("="*70)
        return None
    
    results = {
        'sufficiency': [],
        'faithfulness': [],
        'attention': [],
        'f1': []
    }
    
    print(f"\nAnalyzing {len(items)} samples...")
    print("="*70)
    
    valid_samples = 0
    for item in tqdm(items, desc="Computing metrics"):
        try:
            # Basic F1
            pred = generate(model, tokenizer, item['context'], item['question'], device)
            f1 = f1_score(pred, item['answer'])
            
            # Sufficiency
            suff = compute_sufficiency(model, tokenizer, item, device)
            
            # Faithfulness
            faith = compute_faithfulness(model, tokenizer, item, device)
            
            # Attention on answer tokens
            attn = compute_attention_on_answer_tokens(model, tokenizer, item, device)
            
            # Only include if all metrics were computed
            if suff is not None and faith is not None and attn is not None:
                results['sufficiency'].append(suff)
                results['faithfulness'].append(faith)
                results['attention'].append(attn)
                results['f1'].append(f1)
                valid_samples += 1
            
        except Exception as e:
            print(f"\nError on sample: {e}")
            continue
    
    if valid_samples == 0:
        print("\n" + "="*70)
        print("ERROR: No valid samples were analyzed!")
        print("This usually means the dataset doesn't have answer span info.")
        print("="*70)
        return None
    
    # Compute aggregate statistics
    results['mean_sufficiency'] = np.mean(results['sufficiency'])
    results['mean_faithfulness'] = np.mean(results['faithfulness'])
    results['mean_attention'] = np.mean(results['attention'])
    results['mean_f1'] = np.mean(results['f1'])
    
    results['std_sufficiency'] = np.std(results['sufficiency'])
    results['std_faithfulness'] = np.std(results['faithfulness'])
    results['std_attention'] = np.std(results['attention'])
    
    # Correlations
    if len(results['f1']) > 1:
        results['corr_f1_sufficiency'] = np.corrcoef(results['f1'], results['sufficiency'])[0, 1]
        results['corr_f1_faithfulness'] = np.corrcoef(results['f1'], results['faithfulness'])[0, 1]
        results['corr_f1_attention'] = np.corrcoef(results['f1'], results['attention'])[0, 1]
    
    # Print summary
    print("\n" + "="*70)
    print("RESULTS")
    print("="*70)
    print(f"Samples analyzed: {valid_samples}")
    print(f"\nMean F1: {results['mean_f1']:.4f}")
    print(f"\nSufficiency: {results['mean_sufficiency']:.4f} ± {results['std_sufficiency']:.4f}")
    print(f"  (Can model answer with only answer span?)")
    print(f"\nFaithfulness: {results['mean_faithfulness']:.4f} ± {results['std_faithfulness']:.4f}")
    print(f"  (F1 drop when answer removed - higher = more faithful)")
    print(f"\nAttention on Answer Tokens: {results['mean_attention']:.4f} ± {results['std_attention']:.4f}")
    print(f"  (Average attention on answer token positions)")
    
    if 'corr_f1_attention' in results:
        print(f"\nCorrelations with F1:")
        print(f"  F1 ↔ Sufficiency:   {results['corr_f1_sufficiency']:>6.3f}")
        print(f"  F1 ↔ Faithfulness:  {results['corr_f1_faithfulness']:>6.3f}")
        print(f"  F1 ↔ Attention:     {results['corr_f1_attention']:>6.3f}")
    
    print("="*70)
    
    return results

In [ ]:
# Run this first to load dataset with answer span information
val_dataset_full = load_squad_with_spans('dev-v2.0.json', tokenizer)
print(f"Loaded {len(val_dataset_full)} validation samples with span info")

# 1. Load your baseline model
checkpoint = torch.load('best_baseline.pt')
model.load_state_dict(checkpoint['model'])


In [ ]:
# Now analyze your model
baseline_results = analyze_model(
    model=model,
    dataset=val_dataset_full,  # Use the new dataset!
    tokenizer=tokenizer,
    device=device,
    n_samples=5000
)

In [ ]:
# Run this first to load dataset with answer span information
train_dataset_full = load_squad_with_spans('train-v2.0.json', tokenizer)
print(f"Loaded {len(train_dataset_full)} train samples with span info")

# 1. Load your baseline model
checkpoint = torch.load('best_baseline.pt')
model.load_state_dict(checkpoint['model'])


In [ ]:
# Now analyze your model
train_baseline_results = analyze_model(
    model=model,
    dataset=train_dataset_full,  # Use the new dataset!
    tokenizer=tokenizer,
    device=device,
    n_samples=60000)

In [ ]:
"""
SQuAD Answer Generation with GloVe Embeddings + Q/K Hypothesis Testing

EXPECTED PERFORMANCE:
- With GloVe embeddings: 40-55% F1 ✓
- Training time: ~40-50 minutes
- Can reach 50%+ with Q/K hypothesis
"""

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from transformers import GPT2Tokenizer
import json
from collections import Counter
import string
import re
from tqdm import tqdm
import numpy as np
import os
import urllib.request
import zipfile

# Configuration
TEST_QK_HYPOTHESIS = True  # Set True after baseline completes
QK_LR_MULTIPLIER = 20   # Q/K learn 2.5x faster

# Optimized for GloVe embeddings
D_MODEL = 300  # Match GloVe dimension exactly
N_HEADS = 6
N_LAYERS = 6
D_FF = 1200
MAX_SEQ_LEN = 320
MAX_ANSWER_LEN = 50
DROPOUT = 0.2
BATCH_SIZE = 24
ACCUMULATION_STEPS = 2  # Effective batch: 48
BASE_LR = 5e-5
WARMUP_STEPS = 800
NUM_EPOCHS = 100
GRAD_CLIP = 0.5
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.1
TRAIN_SUBSET_SIZE = 60000  # More data with GloVe
VAL_SUBSET_SIZE = 10000


def download_and_extract_glove():
    """Download and extract GloVe embeddings"""
    glove_file = 'glove.6B.300d.txt'
    
    if os.path.exists(glove_file):
        print(f"✓ GloVe embeddings found: {glove_file}")
        return glove_file
    
    print("\n" + "="*70)
    print("DOWNLOADING GLOVE EMBEDDINGS")
    print("="*70)
    
    zip_file = 'glove.6B.zip'
    
    if not os.path.exists(zip_file):
        print("Downloading GloVe 6B (822MB)... This may take a few minutes")
        url = 'https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip'
        
        try:
            # Download with progress bar
            response = urllib.request.urlopen(url)
            total_size = int(response.headers.get('content-length', 0))
            
            with open(zip_file, 'wb') as f, tqdm(
                total=total_size, unit='B', unit_scale=True, desc='Downloading'
            ) as pbar:
                while True:
                    chunk = response.read(8192)
                    if not chunk:
                        break
                    f.write(chunk)
                    pbar.update(len(chunk))
            
            print("✓ Download complete!")
        except Exception as e:
            print(f"Download failed: {e}")
            print("\nAlternative: Download manually from:")
            print("  https://nlp.stanford.edu/projects/glove/")
            print("  or https://huggingface.co/stanfordnlp/glove")
            return None
    
    # Extract
    if os.path.exists(zip_file):
        print("Extracting GloVe embeddings...")
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            # Only extract the 300d file we need
            zip_ref.extract('glove.6B.300d.txt')
        print("✓ Extraction complete!")
        
        # Optionally remove zip to save space
        # os.remove(zip_file)
    
    if os.path.exists(glove_file):
        return glove_file
    else:
        print("GloVe file not found after extraction")
        return None


def load_glove_embeddings(glove_file, tokenizer, embedding_dim=300):
    """Load GloVe and create embedding matrix for GPT-2 tokenizer"""
    print("\n" + "="*70)
    print("LOADING GLOVE EMBEDDINGS")
    print("="*70)
    
    # Load GloVe vectors
    print("Reading GloVe file (this takes ~1 minute)...")
    glove_vectors = {}
    
    with open(glove_file, 'r', encoding='utf-8') as f:
        for line in tqdm(f, total=400000, desc="Loading GloVe"):
            values = line.rstrip().split(' ')
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            glove_vectors[word] = vector
    
    print(f"✓ Loaded {len(glove_vectors):,} GloVe vectors")
    
    # Create embedding matrix for tokenizer vocabulary
    vocab_size = tokenizer.vocab_size
    embedding_matrix = np.random.normal(0, 0.1, (vocab_size, embedding_dim)).astype('float32')
    
    # Match tokenizer vocab with GloVe
    print("Matching tokenizer vocabulary with GloVe...")
    matched = 0
    
    for token, idx in tqdm(tokenizer.get_vocab().items(), desc="Matching"):
        # Try different matching strategies
        token_clean = token.replace('Ġ', '').replace('Ċ', '').lower().strip()
        
        if token in glove_vectors:
            embedding_matrix[idx] = glove_vectors[token]
            matched += 1
        elif token.lower() in glove_vectors:
            embedding_matrix[idx] = glove_vectors[token.lower()]
            matched += 1
        elif token_clean in glove_vectors:
            embedding_matrix[idx] = glove_vectors[token_clean]
            matched += 1
        # For subword tokens, try averaging character embeddings
        elif len(token_clean) > 0 and all(c.isalpha() for c in token_clean):
            # Use random but consistent embedding for unknown tokens
            pass
    
    match_rate = 100 * matched / vocab_size
    print(f"✓ Matched {matched:,}/{vocab_size:,} tokens ({match_rate:.1f}%)")
    print("="*70 + "\n")
    
    return torch.FloatTensor(embedding_matrix)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.last_attention_weights = None
        
    def forward(self, q, k, v, mask=None, save_attention=False):
        bs = q.size(0)
        
        q = self.q_linear(q).view(bs, -1, self.n_heads, self.d_k).transpose(1, 2)
        k = self.k_linear(k).view(bs, -1, self.n_heads, self.d_k).transpose(1, 2)
        v = self.v_linear(v).view(bs, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.d_k ** 0.5)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn = torch.softmax(scores, dim=-1)
        if save_attention:
            self.last_attention_weights = attn.detach()
        
        attn = self.dropout(attn)
        context = torch.matmul(attn, v)
        context = context.transpose(1, 2).contiguous().view(bs, -1, self.n_heads * self.d_k)
        
        return self.out(context)


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
    def forward(self, x, mask=None, save_attention=False):
        # Pre-norm
        attn_out = self.self_attn(self.norm1(x), self.norm1(x), self.norm1(x), mask, save_attention)
        x = x + attn_out
        x = x + self.ff(self.norm2(x))
        return x


class GPTAnswerGenerator(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, max_seq_len, dropout=0.1, pretrained_embeddings=None):
        super().__init__()
        
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        
        # Initialize with pretrained embeddings if provided
        if pretrained_embeddings is not None:
            print("Initializing token embeddings with GloVe...")
            self.token_embedding.weight.data.copy_(pretrained_embeddings)
            print("✓ Token embeddings initialized with GloVe")
        
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        
        self.norm = nn.LayerNorm(d_model)
        self.output = nn.Linear(d_model, vocab_size)
        
        # Weight tying
        self.output.weight = self.token_embedding.weight
        
        # Initialize non-embedding weights
        self._init_weights()
        
    def _init_weights(self):
        # Don't reinitialize token_embedding if using GloVe
        for name, p in self.named_parameters():
            if 'token_embedding' not in name and p.dim() > 1:
                nn.init.xavier_uniform_(p, gain=1/np.sqrt(2))
        
    def forward(self, x, mask=None, save_attention=False):
        pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        x = self.token_embedding(x) + self.position_embedding(pos)
        x = self.emb_dropout(x)
        
        for layer in self.layers:
            x = layer(x, mask, save_attention)
        
        return self.output(self.norm(x))
    
    def get_attention_weights(self):
        return [layer.self_attn.last_attention_weights for layer in self.layers]


class SQuADDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_len, max_ans_len):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.max_ans_len = max_ans_len
        self.data = []
        
        with open(data_path, 'r') as f:
            squad = json.load(f)
        
        for article in squad['data']:
            for para in article['paragraphs']:
                ctx = para['context']
                for qa in para['qas']:
                    if not qa['is_impossible'] and qa['answers']:
                        ans = qa['answers'][0]['text']
                        ans_start = qa['answers'][0]['answer_start']
                        
                        # Extract relevant context window
                        start = max(0, ans_start - 200)
                        end = min(len(ctx), ans_start + len(ans) + 200)
                        focused_ctx = ctx[start:end]
                        
                        self.data.append({
                            'context': focused_ctx,
                            'question': qa['question'],
                            'answer': ans
                        })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Format: Q: question C: context A: answer
        prefix = f"Q: {item['question']} C: {item['context']} A:"
        answer = f" {item['answer']}"
        
        prefix_ids = self.tokenizer.encode(prefix, max_length=self.max_len-self.max_ans_len-2, 
                                          truncation=True, add_special_tokens=False)
        answer_ids = self.tokenizer.encode(answer, max_length=self.max_ans_len, 
                                          truncation=True, add_special_tokens=False)
        answer_ids.append(self.tokenizer.eos_token_id)
        
        input_ids = prefix_ids + answer_ids
        labels = [-100] * len(prefix_ids) + answer_ids
        
        # Pad
        while len(input_ids) < self.max_len:
            input_ids.append(self.tokenizer.pad_token_id)
            labels.append(-100)
        
        return {
            'input_ids': torch.tensor(input_ids[:self.max_len]),
            'labels': torch.tensor(labels[:self.max_len])
        }


def create_mask(seq_len, device):
    return (torch.triu(torch.ones(seq_len, seq_len, device=device), 1) == 0).unsqueeze(0).unsqueeze(0)


def normalize_answer(s):
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(c for c in s if c not in string.punctuation)
    return ' '.join(s.split())


def f1_score(pred, truth):
    pred_tok = normalize_answer(pred).split()
    truth_tok = normalize_answer(truth).split()
    
    if not pred_tok or not truth_tok:
        return int(pred_tok == truth_tok)
    
    common = Counter(pred_tok) & Counter(truth_tok)
    if not common:
        return 0
    
    prec = sum(common.values()) / len(pred_tok)
    rec = sum(common.values()) / len(truth_tok)
    return 2 * prec * rec / (prec + rec)


def exact_match(pred, truth):
    return int(normalize_answer(pred) == normalize_answer(truth))


def train_epoch(model, loader, opt, sched, device, epoch):
    model.train()
    total_loss = 0
    opt.zero_grad()
    
    pbar = tqdm(loader, desc=f"Epoch {epoch}")
    for i, batch in enumerate(pbar):
        inp = batch['input_ids'].to(device)
        lbl = batch['labels'].to(device)
        
        mask = create_mask(inp.size(1), device)
        logits = model(inp, mask)
        
        # Shift for next-token prediction
        loss = nn.functional.cross_entropy(
            logits[:, :-1].reshape(-1, logits.size(-1)),
            lbl[:, 1:].reshape(-1),
            ignore_index=-100,
            label_smoothing=LABEL_SMOOTHING
        )
        
        loss = loss / ACCUMULATION_STEPS
        loss.backward()
        
        if (i + 1) % ACCUMULATION_STEPS == 0:
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            sched.step()
            opt.zero_grad()
        
        total_loss += loss.item() * ACCUMULATION_STEPS
        pbar.set_postfix({'loss': f'{loss.item() * ACCUMULATION_STEPS:.3f}'})
    
    return total_loss / len(loader)


def generate(model, tokenizer, context, question, device, max_len=50):
    model.eval()
    
    prompt = f"Q: {question} C: {context} A:"
    ids = tokenizer.encode(prompt, max_length=MAX_SEQ_LEN-max_len-5, 
                          truncation=True, add_special_tokens=False, return_tensors='pt').to(device)
    
    start_len = ids.size(1)
    
    with torch.no_grad():
        for _ in range(max_len):
            if ids.size(1) >= MAX_SEQ_LEN:
                break
            
            mask = create_mask(ids.size(1), device)
            logits = model(ids, mask)
            next_tok = logits[:, -1].argmax(-1, keepdim=True)
            ids = torch.cat([ids, next_tok], 1)
            
            if next_tok.item() == tokenizer.eos_token_id:
                break
    
    return tokenizer.decode(ids[0, start_len:], skip_special_tokens=True).strip()


def evaluate(model, dataset, tokenizer, device, n_samples=300):
    model.eval()
    f1_sum = em_sum = 0
    
    if isinstance(dataset, Subset):
        items = [dataset.dataset.data[dataset.indices[i]] for i in range(min(n_samples, len(dataset)))]
    else:
        items = dataset.data[:n_samples]
    
    for item in tqdm(items, desc="Eval"):
        pred = generate(model, tokenizer, item['context'], item['question'], device)
        f1_sum += f1_score(pred, item['answer'])
        em_sum += exact_match(pred, item['answer'])
    
    return {'f1': f1_sum / len(items), 'em': em_sum / len(items)}


def analyze_attention(model, dataset, tokenizer, device, n=30):
    model.eval()
    scores = []
    
    if isinstance(dataset, Subset):
        items = [dataset.dataset.data[dataset.indices[i]] for i in range(min(n, len(dataset)))]
    else:
        items = dataset.data[:n]
    
    for item in items:
        prompt = f"Q: {item['question']} C: {item['context']} A:"
        ids = tokenizer.encode(prompt, max_length=MAX_SEQ_LEN-MAX_ANSWER_LEN, 
                              truncation=True, add_special_tokens=False, return_tensors='pt').to(device)
        
        with torch.no_grad():
            mask = create_mask(ids.size(1), device)
            model(ids, mask, save_attention=True)
            
            weights = model.get_attention_weights()
            if weights[0] is not None:
                avg = torch.stack([w[0] for w in weights if w is not None]).mean(0)
                scores.append(avg[0].mean().item())
    
    return np.mean(scores) if scores else 0


if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print("="*70)
    print("SQUAD ANSWER GENERATION WITH GLOVE EMBEDDINGS")
    print("="*70)
    #print(f"Expected F1: 40-55% (with GloVe)")
    print(f"Model: {N_LAYERS}L, {D_MODEL}d, {N_HEADS}h")
    print(f"Device: {device}")
    print("="*70 + "\n")
    
    # Download and load GloVe
    glove_file = download_and_extract_glove()
    
    if glove_file is None:
        print("\n WARNING: Could not load GloVe embeddings")
        print("Proceeding without pretrained embeddings (expect 15-25% F1)")
        pretrained_embeddings = None
    
    # Download SQuAD datasets
    for name in ['train-v2.0.json', 'dev-v2.0.json']:
        if not os.path.exists(name):
            print(f"Downloading {name}...")
            urllib.request.urlretrieve(
                f'https://rajpurkar.github.io/SQuAD-explorer/dataset/{name}', name)
    
    # Setup tokenizer
    print("Loading tokenizer...")
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')# Assign a custom LR multiplier per layer
# Example: scale grows with layer index (edit as you like)
def qk_lr_scale(layer_idx):
    return 1.0 + 0.2 * layer_idx     # layer0=1.0x, layer1=1.2x, layer2=1.4x, ...

# Collect params
qk_param_groups = []
other = []

for n, p in model.named_parameters():

    if 'q_linear' in n or 'k_linear' in n:

        # extract layer number from name (modify if your naming differs)
        # expected: something like "transformer.layers.3.attn.q_linear.weight"
        layer_id = int([x for x in n.split('.') if x.isdigit()][0])

        qk_param_groups.append({
            'params': [p],
            'lr': BASE_LR * qk_lr_scale(layer_id),
            'weight_decay': WEIGHT_DECAY
        })
    else:
        other.append(p)

print(f"Q/K params: {sum(pg['params'][0].numel() for pg in qk_param_groups)/1e6:.1f}M")
print(f"Other params: {sum(p.numel() for p in other)/1e6:.1f}M\n")

# Build optimizer: all Q/K groups + other params group
opt = torch.optim.AdamW(
    qk_param_groups +
    [{'params': other, 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY}]
)

# Build matching scheduler max_lrs
max_lrs = [pg['lr'] for pg in qk_param_groups] + [BASE_LR]

sched = torch.optim.lr_scheduler.OneCycleLR(
    opt,
    max_lr=max_lrs,
    total_steps=len(loader) * NUM_EPOCHS,
    pct_start=WARMUP_STEPS / (len(loader) * NUM_EPOCHS)
)
    tokenizer.pad_token = tokenizer.eos_token
    
    # Load GloVe embeddings for tokenizer
    if glove_file:
        pretrained_embeddings = load_glove_embeddings(glove_file, tokenizer, D_MODEL)
    else:
        pretrained_embeddings = None
    
    # Load datasets
    print("Loading datasets...")
    full_train = SQuADDataset('train-v2.0.json', tokenizer, MAX_SEQ_LEN, MAX_ANSWER_LEN)
    full_val = SQuADDataset('dev-v2.0.json', tokenizer, MAX_SEQ_LEN, MAX_ANSWER_LEN)
    
    train_ds = Subset(full_train, torch.randperm(len(full_train))[:TRAIN_SUBSET_SIZE])
    val_ds = Subset(full_val, torch.randperm(len(full_val))[:VAL_SUBSET_SIZE])
    
    print(f"Train: {len(train_ds)}, Val: {len(val_ds)}\n")
    
    loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    
    # Model
    print("Initializing model...")
    model = GPTAnswerGenerator(
        vocab_size=tokenizer.vocab_size,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
        d_ff=D_FF,
        max_seq_len=MAX_SEQ_LEN,
        dropout=DROPOUT,
        pretrained_embeddings=pretrained_embeddings
    ).to(device)
    
    total_params = sum(p.numel() for p in model.parameters()) / 1e6
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    print(f"Total parameters: {total_params:.1f}M")
    print(f"Trainable parameters: {trainable_params:.1f}M\n")
    
    # Optimizer with differential learning rates for embeddings
    if TEST_QK_HYPOTHESIS:
        print("="*70)
        print(f"TESTING Q/K HYPOTHESIS - Q/K LR = {QK_LR_MULTIPLIER}x")
        print("="*70 + "\n")
        
        qk = [p for n, p in model.named_parameters() if 'q_linear' in n or 'k_linear' in n]
        other = [p for n, p in model.named_parameters() if 'q_linear' not in n and 'k_linear' not in n]
        
        print(f"Q/K params: {sum(p.numel() for p in qk)/1e6:.1f}M")
        print(f"Other params: {sum(p.numel() for p in other)/1e6:.1f}M\n")
        
        opt = torch.optim.AdamW([
            {'params': qk, 'lr': BASE_LR * QK_LR_MULTIPLIER, 'weight_decay': WEIGHT_DECAY},
            {'params': other, 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY}
        ])
        
        sched = torch.optim.lr_scheduler.OneCycleLR(
            opt, [BASE_LR * QK_LR_MULTIPLIER, BASE_LR],
            total_steps=len(loader) * NUM_EPOCHS,
            pct_start=WARMUP_STEPS / (len(loader) * NUM_EPOCHS)
        )
    else:
        print("="*70)
        print("BASELINE (Standard LR)")
        print("="*70 + "\n")
        
        # Use lower LR for pretrained embeddings if they exist
        if pretrained_embeddings is not None:
            embedding_params = [model.token_embedding.weight]
            other_params = [p for n, p in model.named_parameters() if 'token_embedding' not in n]
            
            opt = torch.optim.AdamW([
                {'params': embedding_params, 'lr': BASE_LR * 0.1, 'weight_decay': 0},  # Fine-tune slowly
                {'params': other_params, 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY}
            ])
            
            print("Using differential LR: embeddings=0.1x, other=1.0x\n")
        else:
            opt = torch.optim.AdamW(model.parameters(), BASE_LR, weight_decay=WEIGHT_DECAY)
        
        sched = torch.optim.lr_scheduler.OneCycleLR(
            opt,
            max_lr=BASE_LR if pretrained_embeddings is None else [BASE_LR * 0.1, BASE_LR],
            total_steps=len(loader) * NUM_EPOCHS,
            pct_start=WARMUP_STEPS / (len(loader) * NUM_EPOCHS)
        )
    
    # Train
    best_f1 = 0
    results = {'loss': [], 'train_f1': [], 'val_f1': [], 'val_em': [], 'attn': []}
    
    for e in range(NUM_EPOCHS):
        print(f"\n{'='*70}")
        print(f"EPOCH {e+1}/{NUM_EPOCHS}")
        print('='*70)
        
        loss = train_epoch(model, loader, opt, sched, device, e+1)
        results['loss'].append(loss)
        print(f"\nLoss: {loss:.4f}")
        
        # Eval
        train_m = evaluate(model, train_ds, tokenizer, device, 200)
        val_m = evaluate(model, val_ds, tokenizer, device, 300)
        
        results['train_f1'].append(train_m['f1'])
        results['val_f1'].append(val_m['f1'])
        results['val_em'].append(val_m['em'])
        
        gap = train_m['f1'] - val_m['f1']
        print(f"Train F1: {train_m['f1']:.4f} | Val F1: {val_m['f1']:.4f} | Gap: {gap:.4f} | EM: {val_m['em']:.4f}")
        
        # Sample
        if e % 2 == 0:
            item = val_ds.dataset.data[val_ds.indices[0]]
            pred = generate(model, tokenizer, item['context'], item['question'], device)
            print(f"\nSample:")
            print(f"  Q: {item['question'][:60]}...")
            print(f"  True: {item['answer']}")
            print(f"  Pred: {pred}")
            print(f"  F1: {f1_score(pred, item['answer']):.3f}")
        
        # Attention
        if e % 4 == 0 and TEST_QK_HYPOTHESIS:
            attn = analyze_attention(model, val_ds, tokenizer, device)
            results['attn'].append(attn)
            print(f"Attention: {attn:.4f}")
        
        # Save best
        if val_m['f1'] > best_f1:
            best_f1 = val_m['f1']
            name = 'best_qk_20x.pt' if TEST_QK_HYPOTHESIS else 'best_baseline.pt'
            torch.save({'model': model.state_dict(), 'f1': best_f1, 'epoch': e+1}, name)
            print(f"✓ SAVED! Best F1: {best_f1:.4f}")
    
    # Final
    print(f"\n{'='*70}")
    print("FINAL RESULTS")
    print('='*70)
    print(f"Best Val F1: {best_f1*100:.1f}%")
    print(f"Final Val F1: {results['val_f1'][-1]*100:.1f}%")
    print(f"Final EM: {results['val_em'][-1]*100:.1f}%")
    print(f"Train-Val Gap: {results['train_f1'][-1] - results['val_f1'][-1]:.4f}")
    
    if pretrained_embeddings is not None:
        if best_f1 >= 0.40:
            print(f"\n✓ EXCELLENT! Hit target with GloVe embeddings!")
        elif best_f1 >= 0.30:
            print(f"\n✓ GOOD! GloVe embeddings helping significantly")
        else:
            print(f"\n⚠ Below expected (40%+) - may need more training")
    
    print('='*70)
    
    name = 'results_qk_glove.pt' if TEST_QK_HYPOTHESIS else 'results_baseline_glove.pt'
    torch.save(results, name)
    print(f"\n✓ Saved to {name}")
    
    if not TEST_QK_HYPOTHESIS and best_f1 > 0.30:
        print(f"\n{'='*70}")
        print("BASELINE COMPLETE!")
        print("Now set TEST_QK_HYPOTHESIS=True to test your hypothesis")
        print(f"Target: Beat {best_f1*100:.1f}% F1 with Q/K boosted learning")
        print('='*70)

In [ ]:
# Run this first to load dataset with answer span information
val_dataset_full = load_squad_with_spans('dev-v2.0.json', tokenizer)
print(f"Loaded {len(val_dataset_full)} validation samples with span info")

# 1. Load your baseline model
checkpoint = torch.load('best_qk_20x.pt')
model.load_state_dict(checkpoint['model'])

In [ ]:
# Now analyze your model
baseline_results = analyze_model(
    model=model,
    dataset=val_dataset_full,  # Use the new dataset!
    tokenizer=tokenizer,
    device=device,
    n_samples=5000
)

In [ ]:
# Run this first to load dataset with answer span information
train_dataset_full = load_squad_with_spans('train-v2.0.json', tokenizer)
print(f"Loaded {len(train_dataset_full)} train samples with span info")

# 1. Load your baseline model
checkpoint = torch.load('best_qk_20x.pt')
model.load_state_dict(checkpoint['model'])

In [ ]:
# Now analyze your model
baseline_results_train = analyze_model(
    model=model,
    dataset=train_dataset_full,  # Use the new dataset!
    tokenizer=tokenizer,
    device=device,
    n_samples=60000
)